# Stateful 대화 (대화 기록 유지)

`1_6_stateless.ipynb`에서 보았듯이 LLM API는 **stateless**입니다.
요청 사이에 서버가 아무것도 기억하지 않기 때문에, 이전 대화를 기억하게 하려면
**매 요청마다 지금까지의 대화 전체를 messages에 다시 담아 보내야** 합니다.

이 노트북에서는 stateless를 stateful로 바꾸는 두 가지 방법을 봅니다.

1. `messages` 리스트에 **모델의 답변까지** 직접 누적하기
2. Gradio `ChatInterface`가 넘겨주는 `history`를 사용하기

In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

import os

load_dotenv(override=True)
# OpenAI 크레딧이 없어서 Gemini 로 보냅니다. (OpenAI 호환 주소 사용 - 1_2_gemini 참고)
# 크레딧을 충전하면 openai = OpenAI() 로 되돌리고 MODEL 을 gpt-5.4-nano 등으로 바꾸세요.
openai = OpenAI(api_key=os.getenv("GOOGLE_API_KEY"), base_url=os.getenv("GEMINI_URL"))
MODEL = "gemini-3.1-flash-lite"  # gemini-3.8-flash 는 무료 한도가 하루 20회라 lite 사용

d:\a_i\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
system_prompt = """
당신은 대화 전문가이며 대화하는 사람의 고민을 들어주고 제안을 할 수 있습니다.
"""

## 1. messages에 직접 누적하기

핵심은 **assistant의 답변도 messages에 추가**하는 것입니다.
stateless 예제에서는 user 메시지만 보냈기 때문에 모델이 자기가 방금 한 말조차 몰랐습니다.

In [3]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "나의 이름은 이상화입니다."}
]

In [4]:
response = openai.chat.completions.create(model=MODEL, messages=messages)
reply = response.choices[0].message.content

# 모델의 답변을 대화 기록에 추가 (이것이 stateful의 핵심)
messages.append({"role": "assistant", "content": reply})
print(reply)

반갑습니다, 상화 님! 이렇게 인연을 맺게 되어 정말 기쁩니다.

저는 사람들의 이야기를 듣고, 함께 고민하며 조금 더 나은 방향을 찾아가는 대화 전문가입니다. 상화라는 이름이 참 따뜻하고 단단하게 느껴지네요.

지금 상화 님 마음속에 어떤 고민이나 이야기가 자리 잡고 있나요? 거창한 고민이 아니어도 좋아요. 오늘 하루 있었던 소소한 일부터, 누군가에게 쉽게 털어놓지 못했던 깊은 속마음까지 무엇이든 좋습니다.

편안하게 말씀해 주세요. 제가 잘 듣고, 함께 고민해 드릴게요. 무엇부터 이야기해 볼까요?


In [5]:
# 다음 질문도 같은 messages 리스트에 이어서 추가합니다
messages.append({"role": "user", "content": "내 이름은 무엇인가요?"})

response = openai.chat.completions.create(model=MODEL, messages=messages)
reply = response.choices[0].message.content

messages.append({"role": "assistant", "content": reply})
print(reply)

방금 말씀해 주신 상화 님의 성함은 **'이상화'**입니다. 

혹시 이름에 대해 궁금한 점이 있으시거나, 이름과 관련해 떠오르는 생각이 있으신가요? 말씀해 주시면 더 즐겁게 대화를 이어가 볼게요!


In [6]:
# 지금까지 쌓인 대화 기록 확인
for m in messages:
    print(f"[{m['role']}] {m['content']}")

[system] 
당신은 대화 전문가이며 대화하는 사람의 고민을 들어주고 제안을 할 수 있습니다.

[user] 나의 이름은 이상화입니다.
[assistant] 반갑습니다, 상화 님! 이렇게 인연을 맺게 되어 정말 기쁩니다.

저는 사람들의 이야기를 듣고, 함께 고민하며 조금 더 나은 방향을 찾아가는 대화 전문가입니다. 상화라는 이름이 참 따뜻하고 단단하게 느껴지네요.

지금 상화 님 마음속에 어떤 고민이나 이야기가 자리 잡고 있나요? 거창한 고민이 아니어도 좋아요. 오늘 하루 있었던 소소한 일부터, 누군가에게 쉽게 털어놓지 못했던 깊은 속마음까지 무엇이든 좋습니다.

편안하게 말씀해 주세요. 제가 잘 듣고, 함께 고민해 드릴게요. 무엇부터 이야기해 볼까요?
[user] 내 이름은 무엇인가요?
[assistant] 방금 말씀해 주신 상화 님의 성함은 **'이상화'**입니다. 

혹시 이름에 대해 궁금한 점이 있으시거나, 이름과 관련해 떠오르는 생각이 있으신가요? 말씀해 주시면 더 즐겁게 대화를 이어가 볼게요!


## 2. 함수로 정리하기

매번 append를 직접 쓰는 대신 함수로 감싸면 편합니다.
`conversation` 리스트가 대화 상태(state)를 들고 있습니다.

In [7]:
conversation = [{"role": "system", "content": system_prompt}]

def ask(message, model=MODEL):
    conversation.append({"role": "user", "content": message})
    response = openai.chat.completions.create(model=model, messages=conversation)
    reply = response.choices[0].message.content
    conversation.append({"role": "assistant", "content": reply})
    return reply

def reset():
    """대화 기록을 초기화합니다."""
    global conversation
    conversation = [{"role": "system", "content": system_prompt}]

In [8]:
print(ask("나의 이름은 이상화입니다."))

반갑습니다, 상화 님! 이렇게 인연이 닿아 대화를 나누게 되어 정말 기쁩니다.

저는 상화 님의 이야기를 진심으로 들어드리고, 때로는 함께 고민하며 더 나은 방향을 찾도록 돕는 대화 상대가 되어드릴게요.

오늘 어떤 이야기를 나누고 싶으신가요? 마음속에 담아두었던 고민이나, 혹은 그냥 누군가에게 털어놓고 싶은 일상적인 이야기라도 괜찮습니다. 무엇이든 편하게 말씀해 주세요. 준비되셨을 때 시작할까요?


In [9]:
print(ask("내 이름은 무엇인가요?"))

상화 님의 성함은 **이상화** 님입니다.

조금 전에 저에게 직접 말씀해 주셨죠! 기억하고 있습니다. 무엇을 도와드릴까요?


In [10]:
print(f"대화 기록 개수: {len(conversation)}")

대화 기록 개수: 5


## 3. Gradio로 stateful 챗봇 만들기

`gr.ChatInterface`는 이전 대화를 `history` 인자로 넘겨줍니다.
`history`는 `[{"role": ..., "content": ...}, ...]` 형태로, OpenAI의 messages 형식과 같습니다.
그래서 변환 없이 messages에 바로 이어 붙일 수 있습니다.

stateless 버전과 비교해 보세요. 차이는 `+ history` 한 군데뿐입니다.

```python
# stateless
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": message}]

# stateful
messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
```

In [11]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

In [12]:
dark_css = """
.gradio-container, .gradio-container .fillable, body {
    background-color: #000000 !important;
}
"""

with gr.Blocks() as chat_ui:
    gr.ChatInterface(fn=chat, title="With history")

chat_ui.launch(css=dark_css, inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## 4. 주의할 점

대화가 길어질수록 매 요청에 보내는 messages가 계속 커집니다.

- **비용/지연 증가**: 입력 토큰이 매 턴 누적됩니다.
- **컨텍스트 한계**: 모델의 최대 컨텍스트를 넘으면 오류가 납니다.

그래서 실제 서비스에서는 오래된 대화를 잘라내거나(truncation),
요약해서 보관하는(summarization) 방법을 씁니다.
이 내용은 `1_7_context_openai.ipynb`에서 이어집니다.